In [ ]:
"""LANGHAM Hotel Management System.

Student Name: Liriks Yadav
Student ID: 850009620
Date: 05/11/2026

"""

Task 1

In [1]:

from __future__ import annotations

import ast
from dataclasses import dataclass
from datetime import datetime
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from pathlib import Path

STUDENT_ID = "850009620" 
GST_RATE = Decimal("0.15")
MAX_NIGHTS = 365
DEFAULT_RATES = {"Standard": Decimal("160.00"),
                 "Deluxe": Decimal("240.00"),
                 "Suite": Decimal("360.00")}
PENNY = Decimal("0.01")


@dataclass(frozen=True)
class Room:
    """A room and its pricing details; current status comes from bookings."""

    number: int
    category: str
    nightly_rate: Decimal


@dataclass(frozen=True)
class Customer:
    """Details of a customer staying in a room."""

    customer_number: int
    full_name: str


@dataclass(frozen=True)
class Allocation:
    """The connection between one room, a customer, and a stay."""

    room_number: int
    customer: Customer
    nights: int
    checked_in: str


class Hotel:
    """Room inventory, active allocations, billing, and text-file operations."""

    def __init__(self, folder: Path, student_id: str = STUDENT_ID):
        self.folder = Path(folder)
        self.student_id = str(student_id).strip()
        if not self.student_id:
            raise ValueError("A student ID is required for file naming.")
        self.rooms: dict[int, Room] = {}
        self.allocations: dict[int, Allocation] = {}
        self.invoices: list[dict] = []
        self.rates = self.load_rates()

    @property
    def allocation_file(self) -> Path:
        return self.folder / f"LHMS_{self.student_id}.txt"

    def load_rates(self) -> dict[str, Decimal]:
        """Read optional literal configuration, or fall back to safe defaults."""
        path = self.folder / "room_rates.txt"
        try:
            with path.open("r", encoding="utf-8") as handle:
                contents = handle.read()
            raw = ast.literal_eval(contents)
            if not isinstance(raw, dict):
                raise TypeError("Room-rate configuration must be a dictionary.")
            rates = {}
            for category, amount in raw.items():
                if not isinstance(category, str) or not category.strip():
                    raise TypeError("Category names must be nonempty text.")
                if isinstance(amount, bool) or not isinstance(amount, (int, float, str)):
                    raise TypeError("Rates must be numeric values or numeric text.")
                try:
                    rate = Decimal(str(amount))
                except InvalidOperation as exc:
                    raise ValueError("A room rate is not numeric.") from exc
                if not rate.is_finite() or rate <= 0:
                    raise ValueError("Room rates must be positive and finite.")
                rates[category] = rate.quantize(PENNY, rounding=ROUND_HALF_UP)
            if not rates:
                raise ValueError("The room-rate list cannot be empty.")
            return rates
        except FileNotFoundError:
            return DEFAULT_RATES.copy()  # Optional configuration: not an error.
        except (SyntaxError, ValueError, TypeError, OSError) as error:
            print(f"Rate configuration ignored ({type(error).__name__}); using defaults.")
            return DEFAULT_RATES.copy()

    def add_room(self, number: int, category: str) -> Room:
        """Add one distinct room without resetting existing inventory."""
        if not isinstance(number, int) or isinstance(number, bool):
            raise TypeError("Room number must be a whole number.")
        if number <= 0:
            raise ValueError("Room number must be positive.")
        if number in self.rooms:
            raise ValueError("That room number already exists.")
        if category not in self.rates:
            raise ValueError("Unknown room category.")
        room = Room(number, category, self.rates[category])
        self.rooms[number] = room
        return room

    def delete_room(self, number: int) -> None:
        """Remove an existing room only when it has no active allocation."""
        if number not in self.rooms:
            raise ValueError("Room not found.")
        if number in self.allocations:
            raise ValueError("Cannot delete an occupied room.")
        del self.rooms[number]

    def availability(self, number: int) -> str:
        return "Occupied" if number in self.allocations else "Available"

    def occupancy_percentage(self) -> Decimal:
        """Return occupancy, including the zero-room boundary case."""
        if not self.rooms:
            return Decimal("0.00")
        try:
            return (Decimal(len(self.allocations)) / Decimal(len(self.rooms))
                    * 100).quantize(PENNY, rounding=ROUND_HALF_UP)
        except (ZeroDivisionError, InvalidOperation):
            return Decimal("0.00")

    def allocate(self, room_number: int, customer_number: int,
                 customer_name: str, nights: int) -> Allocation:
        """Book an available room after checking all customer and stay inputs."""
        if room_number not in self.rooms:
            raise ValueError("Room not found.")
        if room_number in self.allocations:
            raise ValueError("Room is already occupied.")
        if not isinstance(customer_number, int) or customer_number <= 0:
            raise ValueError("Customer number must be positive.")
        if not isinstance(customer_name, str) or not customer_name.strip():
            raise ValueError("Customer name cannot be empty.")
        if not isinstance(nights, int) or isinstance(nights, bool):
            raise TypeError("Nights must be a whole number.")
        if not 1 <= nights <= MAX_NIGHTS:
            raise ValueError(f"Nights must be between 1 and {MAX_NIGHTS}.")
        # Customer IDs may repeat for the same name, but cannot identify
        # two different people in simultaneous active allocations.
        for existing in self.allocations.values():
            if existing.customer.customer_number == customer_number and (
                    existing.customer.full_name.casefold() != customer_name.strip().casefold()):
                raise ValueError("Customer ID already belongs to a different guest.")
        allocation = Allocation(room_number, Customer(customer_number, customer_name.strip()),
                                nights, datetime.now().strftime("%d/%m/%Y %H:%M"))
        self.allocations[room_number] = allocation
        return allocation

    def bill_and_release(self, room_number: int) -> dict:
        """Calculate a complete GST bill before releasing a room."""
        if room_number not in self.allocations:
            raise ValueError("This room has no active allocation.")
        booking = self.allocations[room_number]
        room = self.rooms[room_number]
        try:
            if not isinstance(room.nightly_rate, Decimal):
                raise TypeError("Invalid room-rate data")
            subtotal = (room.nightly_rate * booking.nights).quantize(PENNY, rounding=ROUND_HALF_UP)
            gst = (subtotal * GST_RATE).quantize(PENNY, rounding=ROUND_HALF_UP)
            total = (subtotal + gst).quantize(PENNY, rounding=ROUND_HALF_UP)
        except (TypeError, InvalidOperation, OverflowError) as exc:
            # If an invoice cannot be produced, do not remove the allocation.
            raise ValueError("Could not calculate the bill; room remains occupied.") from exc
        invoice = {"room": room_number, "guest": booking.customer.full_name,
                   "customer_number": booking.customer.customer_number,
                   "nights": booking.nights, "rate": room.nightly_rate,
                   "subtotal": subtotal, "gst": gst, "total": total,
                   "issued": datetime.now().strftime("%d/%m/%Y %H:%M")}
        self.invoices.append(invoice)
        del self.allocations[room_number]
        return invoice

    def format_allocations(self) -> str:
        lines = ["LANGHAM HOTEL - ACTIVE ROOM ALLOCATIONS", "=" * 48]
        if not self.allocations:
            lines.append("No rooms are currently allocated.")
        for number in sorted(self.allocations):
            booking = self.allocations[number]
            room = self.rooms[number]
            lines.extend([f"Room: {number} ({room.category})",
                          f"Guest: {booking.customer.full_name} (ID {booking.customer.customer_number})",
                          f"Nights: {booking.nights}", f"Rate/Night: NZ${room.nightly_rate:.2f}",
                          f"Check-in: {booking.checked_in}", "-" * 48])
        return "\n".join(lines) + "\n"

    def save_allocations(self) -> Path:
        """Create or overwrite the current text file, then close it."""
        try:
            self.folder.mkdir(parents=True, exist_ok=True)
            with self.allocation_file.open("w", encoding="utf-8") as handle:
                handle.write(self.format_allocations())
        except OSError as exc:  # IOError is an alias of OSError in Python 3.
            raise IOError(f"Unable to save allocation file: {exc}") from exc
        return self.allocation_file

    def read_allocations_file(self) -> str:
        """Read the saved allocation file without modifying it."""
        try:
            with self.allocation_file.open("r", encoding="utf-8") as handle:
                return handle.read()
        except FileNotFoundError:
            raise FileNotFoundError("No saved file exists. Use option 7 first.") from None
        except OSError as exc:
            raise IOError(f"Unable to read allocation file: {exc}") from exc

    def backup_and_clear(self) -> Path:
        """Append to a timestamped backup, then clear only the saved text file."""
        content = self.read_allocations_file()
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        backup = self.folder / f"LHMS_{self.student_id}_Backup_{stamp}.txt"
        try:
            with backup.open("a", encoding="utf-8") as handle:
                handle.write(content)
            # Clear only after the backup write has succeeded.
            with self.allocation_file.open("w", encoding="utf-8") as handle:
                handle.write("")
        except OSError as exc:
            raise IOError(f"Backup or clear failed: {exc}") from exc
        return backup


def get_integer(prompt: str, minimum: int = 1, maximum: int | None = None) -> int:
    """Repeat until the user supplies a whole number in the allowed range."""
    while True:
        try:
            value = int(input(prompt).strip())
            if value < minimum or (maximum is not None and value > maximum):
                raise ValueError("Outside the permitted range.")
            return value
        except ValueError:
            label = f"{minimum} or greater" if maximum is None else f"{minimum} to {maximum}"
            print(f"Invalid number. Enter a whole number from {label}.")


def get_name(prompt: str) -> str:
    while True:
        name = input(prompt).strip()
        if name and any(letter.isalpha() for letter in name):
            return name
        print("Please enter a customer name containing letters.")


def choose_category(hotel: Hotel) -> str:
    options = list(hotel.rates.items())
    for index, (name, rate) in enumerate(options, start=1):
        print(f"  {index}. {name} - NZ${rate:.2f} per night")
    while True:
        number = get_integer("Choose room category: ", minimum=1)
        try:
            return options[number - 1][0]
        except IndexError:
            print("That room category does not exist. Please choose again.")


def print_rooms(hotel: Hotel) -> None:
    print("\nROOM DETAILS")
    if not hotel.rooms:
        print("There are no rooms. Add rooms using option 1.")
        return
    print(f"{'Room':<8}{'Category':<14}{'Rate (NZ$)':<15}{'Status'}")
    for number in sorted(hotel.rooms):
        room = hotel.rooms[number]
        print(f"{number:<8}{room.category:<14}{str(room.nightly_rate):<15}{hotel.availability(number)}")
    print(f"Total: {len(hotel.rooms)} | Occupied: {len(hotel.allocations)} | "
          f"Occupancy: {hotel.occupancy_percentage():.2f}%")


def show_menu() -> None:
    print("\n" + "=" * 61)
    print("       LANGHAM HOTEL MANAGEMENT SYSTEM - PYTHON")
    print("=" * 61)
    menu = ("1. Add Rooms", "2. Delete Rooms", "3. Display Rooms Details",
            "4. Allocate Rooms", "5. Display Room Allocation Details",
            "6. Billing & De-Allocation", "7. Save Room Allocation to a File",
            "8. Display Room Allocation from the File",
            "9. Backup and Clear Room Allocation File", "0. Exit Application")
    for line in menu:
        print(line)


def perform_choice(hotel: Hotel, choice: int) -> None:
    """Execute one menu operation. Input validation happens in helpers."""
    if choice == 1:
        count = get_integer("How many rooms would you like to add? ")
        for _ in range(count):
            number = get_integer("Room number: ")
            category = choose_category(hotel)
            try:
                hotel.add_room(number, category)
                print(f"Room {number} ({category}) added.")
            except (ValueError, TypeError) as exc:
                print(f"Could not add room: {exc}")
    elif choice == 2:
        count = get_integer("How many rooms would you like to delete? ")
        for _ in range(count):
            number = get_integer("Room number to delete: ")
            try:
                hotel.delete_room(number)
                print(f"Room {number} deleted.")
            except ValueError as exc:
                print(f"Could not delete: {exc}")
    elif choice == 3:
        print_rooms(hotel)
    elif choice == 4:
        number = get_integer("Room number to allocate: ")
        if number not in hotel.rooms or number in hotel.allocations:
            print("That room does not exist or is already occupied.")
            return
        customer_number = get_integer("Customer number: ")
        customer_name = get_name("Customer name: ")
        nights = get_integer("Number of nights (1-365): ", maximum=MAX_NIGHTS)
        try:
            hotel.allocate(number, customer_number, customer_name, nights)
            print(f"Room {number} allocated to {customer_name} for {nights} night(s).")
        except (ValueError, TypeError) as exc:
            print(f"Booking refused: {exc}")
    elif choice == 5:
        print("\n" + hotel.format_allocations())
    elif choice == 6:
        number = get_integer("Room number to check out: ")
        try:
            bill = hotel.bill_and_release(number)
            print("\n--- CUSTOMER INVOICE ---")
            print(f"Customer: {bill['guest']} | Room: {bill['room']}")
            print(f"{bill['nights']} night(s) x NZ${bill['rate']:.2f}")
            print(f"Subtotal: NZ${bill['subtotal']:.2f}")
            print(f"GST (15%): NZ${bill['gst']:.2f}")
            print(f"Total Due: NZ${bill['total']:.2f}")
            print("Room released for the next guest.")
        except ValueError as exc:
            print(f"Checkout refused: {exc}")
    elif choice == 7:
        print(f"Saved current allocations to {hotel.save_allocations().name}.")
    elif choice == 8:
        print("\n" + hotel.read_allocations_file())
    elif choice == 9:
        print(f"Backup created: {hotel.backup_and_clear().name}")
        print(f"Original file {hotel.allocation_file.name} is now empty.")


def main() -> None:
    hotel = Hotel(Path.cwd())
    while True:
        show_menu()
        try:
            choice = get_integer("Enter your choice (0-9): ", minimum=0, maximum=9)
            if choice == 0:
                print("Thank you for using the LANGHAM Hotel Management System.")
                break
            perform_choice(hotel, choice)
        except (EOFError, KeyboardInterrupt):
            print("\nInput ended. Closing the application safely.")
            break
        except FileNotFoundError as exc:
            print(f"Missing file: {exc}")
        except IOError as exc:
            print(f"File operation failed: {exc}")
        except NameError as exc:
            # A programmer defect; report it instead of displaying a raw traceback.
            print(f"Unexpected function or name error: {exc}")


if __name__ == "__main__":
    main()



       LANGHAM HOTEL MANAGEMENT SYSTEM - PYTHON
1. Add Rooms
2. Delete Rooms
3. Display Rooms Details
4. Allocate Rooms
5. Display Room Allocation Details
6. Billing & De-Allocation
7. Save Room Allocation to a File
8. Display Room Allocation from the File
9. Backup and Clear Room Allocation File
0. Exit Application


Enter your choice (0-9):  1
How many rooms would you like to add?  2
Room number:  001


  1. Standard - NZ$160.00 per night
  2. Deluxe - NZ$240.00 per night
  3. Suite - NZ$360.00 per night


Choose room category:  1


Room 1 (Standard) added.


Room number:  002


... (23 outputs left out)

Enter your choice (0-9):  9


Backup created: LHMS_850009620_Backup_20261106_145251_551509.txt
Original file LHMS_850009620.txt is now empty.

       LANGHAM HOTEL MANAGEMENT SYSTEM - PYTHON
1. Add Rooms
2. Delete Rooms
3. Display Rooms Details
4. Allocate Rooms
5. Display Room Allocation Details
6. Billing & De-Allocation
7. Save Room Allocation to a File
8. Display Room Allocation from the File
9. Backup and Clear Room Allocation File
0. Exit Application


Enter your choice (0-9):  0


Thank you for using the LANGHAM Hotel Management System.
